In [1]:
%matplotlib inline

import os
import sys
import copy

import torch
import numpy as np

import matplotlib.pyplot as plt

from pathlib import Path

sys.path.append('../../../')

%load_ext autoreload
%autoreload 2

from computer_vision.yolov11_pose.parameter_parser import parser
from computer_vision.yolov11_pose.nn.tasks import PoseModel
from computer_vision.yolov11_pose.utils.metrics import DetMetrics, PoseMetrics, box_iou
from computer_vision.yolov11_pose.engine.validator import DectectionValidator
from computer_vision.yolov11_pose.model.validator import PoseValidator
from computer_vision.yolov11_pose.utils.checks import check_imgsz
from computer_vision.yolov11_pose.data.utils import check_det_dataset
from computer_vision.yolov11_pose.data.build import build_yolo_dataset, build_dataloader
from computer_vision.yolov11_pose.cfg import get_cfg
from computer_vision.yolov11_pose.utils.torch_utils import unwrap_model
from computer_vision.yolov11_pose.utils.nms import non_max_suppression # for post processing
from computer_vision.yolov11_pose.utils.ops import xywh2xyxy, xyxy2xywh

In [2]:
main_dirpath='D:/results/yolov11_pose'
checkpoint_dirpath=os.path.join(main_dirpath, 'predict')
output_dirpath=os.path.join(main_dirpath, 'validation')
args=parser.parse_args(f'--save-dir {output_dirpath} --data ../coco8-pose.yaml'.split())


task='pose'
if task=='pose':
    data_dirpath='D:/data/ultralytics/coco8-pose/images/val'
    data_config='../coco8-pose.yaml'
elif task=='detect':
    data_dirpath='D:/data/ultralytics/coco/images/val2017'
    data_config='../coco.yaml'
elif task=='segment':
    data_dirpath='D:/data/ultralytics/coco8-seg/images/val'
    data_config='../coco8-seg.yaml'



hyp=get_cfg()
dataset=build_yolo_dataset(args=args, cfg=hyp, task=task, img_path=data_dirpath, batch=args.batch_size or hyp.batch, 
                   data=data_config,  mode='val', rect=False, stride=32, channels=3)
for i in range(4): print(dataset.labels[i]['cls'].shape)
dataloader=build_dataloader(dataset, batch=hyp.batch, workers=1, shuffle=True, drop_last=True, pin_memory=True)

validator=PoseValidator(dataloader=dataloader, save_dir=args.save_dir, args=args)
validator.args.save_dir

In data.dataset.YOLODataset.get_labels cache_path D:\data\ultralytics\coco8-pose\labels\val.cache
Scanning D:\data\ultralytics\coco8-pose\labels\val.cache... 4 images, 0 backgrounds, 0 corrupts
(8, 1)
(3, 1)
(2, 1)
(1, 1)


'D:/results/yolov11_pose\\validation'

In [3]:
cfg='../yolo11-pose.yaml'
model=PoseModel(cfg=cfg,nc=1,verbose=True)
model.fuse(); # we need to call `fuse` so we can successfully load pretrained weight
checkpoint_file=os.path.join(checkpoint_dirpath, 'yolo11n_torch.pt')
assert os.path.isfile(checkpoint_file), f'{checkpoint_file} does not exist'
checkpoint=torch.load(checkpoint_file, weights_only=False)
try:
    model.load_state_dict(checkpoint['model'])
except RuntimeError as err:
    state_dict=copy.deepcopy(model.state_dict())
    for name, params in checkpoint['model'].items():
        name=name[len('model.'):] # each parameter name is model.model.xxx so we need to remove 1 model.
        
        if name not in state_dict or params.shape!=state_dict[name].shape:
            print(name, name in state_dict, (params.shape,state_dict[name].shape) if name in state_dict else None)
        else: state_dict[name]=params
    model.load_state_dict(state_dict)

In nn.tasks.DetectionModel.__init__ input 1 is not equal to nc in yaml 80->overwrite yaml by input
In nn.tasks.parse_model nc 1, act None, scales {'n': [0.5, 0.25, 1024], 's': [0.5, 0.5, 1024], 'm': [0.5, 1.0, 512], 'l': [1.0, 1.0, 512], 'x': [1.0, 1.5, 512]}
In nn.tasks.parse_model depth 1.0, width 1.0, kpt_shape [17, 3]
In nn.tasks.parse_model no model scale passed. Assuming scale=n.
In nn.tasks.parse_model depth 0.5, width 0.25, max_channels 1024

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  Conv                                         [3, 16, 3, 2]                 
  1                  -1  1      4672  Conv                                         [16, 32, 3, 2]                
  2                  -1  1      6640  C3k2                                         [32, 64, 1, False, 0.25]      
  3                  -1  1     36992  Conv                                         [64, 6

In [4]:
trainer=None
# def __call__(self, trainer=None, model=None):
validator.training=trainer is not None
augment=validator.args.augment and (not validator.training)

if validator.training:
    validator.device=trainer.device
    validator.data=trainer.data
    model=trainer.model # or trainer.ema.ema
    if trainer.args.compile and hasattr(model, '_orig_mod'): model=model._orig_mod # validate non-compiled original model to avoid issues
    validator.loss=torch.zeros_like(trainer.loss_items, device=trainer.device)
    validator.args.plots &= trainer.stopper.possible_stop or (trainer.epoch==trainer.epochs-1)
    model.eval()
else:
    assert model is not None, f'Please provide model'
    assert validator.dataloader is not None, f'Please provide dataloader when initialize validator'
    validator.device=list(model.parameters())[0].device
    stride=model.stride
    imgsz=check_imgsz(validator.args.imgsz, stride=stride)
    if str(validator.args.data).rsplit('.', 1)[-1] in {'yaml', 'yml'}:
        validator.data=check_det_dataset(validator.args.data)
    # elif validator.args.task=='classify':
    #     validator.data=check_cls_dataset(validator.args.data, split=validator.args.split)
    else:raise FileNotFoundError(f'Dataset {validator.args.data} for task={validator.args.task} is not found')

    if validator.device.type in {'cpu', 'mps'}: validator.args.workers=0 # faster CPU val as time dominated by inference, not dataloading
    validator.stride=model.stride # used in get_dataloader() for padding
    model.eval()
validator.init_metrics(unwrap_model(model))
validator.jdict=[] # empty before each val

In [5]:
for batch_i, batch in enumerate(validator.dataloader):
    validator.batch_i=batch_i

    # Preprocessing
    batch=validator.preprocess(batch)

    with torch.no_grad():
        # Inference
        preds=model(batch['img'], augment=augment)
    
        # Loss
        if validator.training: validator.loss+=model.loss(batch, preds)[1]

    # Postprocess
    preds=validator.postprocess(preds)
    break

In [6]:
validator.args.save_json=validator.args.save_txt=True
# def update_metrics(self, preds:list[dict[str, torch.Tensor]], batch:dict[str, Any])->None:
"""Update metrics with new predictions and ground truth

Args:
    preds (list[dict[str, torch.Tensor]]): List of predictions from the model
    batch (dict[str, Any]): Batch data containing ground truth
"""
for si, pred in enumerate(preds):
    validator.seen+=1
    pbatch=validator._prepare_batch(si, batch)
    predn=validator._prepare_pred(pred)

    cls=pbatch['cls'].cpu().numpy()
    no_pred=predn['cls'].shape[0]==0
    # pass numpy array inputs
    validator.metrics.update_stats({**validator._process_batch(predn, pbatch), # tp:(M,10) where 10 is the number of IoU thresholds
                               "target_cls":cls, # (N,)
                               "target_img":np.unique(cls),
                               "conf":np.zeros(0) if no_pred else predn["conf"].cpu().numpy(), # (M,)
                               "pred_cls":np.zeros(0) if no_pred else predn["cls"].cpu().numpy(), # (M,)
                               })
    # Evaluate
    if validator.args.plots:
        validator.confusion_matrix.process_batch(predn, pbatch, conf=validator.args.conf)
        if validator.args.visualize:
            validator.confusion_matrix.plot_matches(batch['img'][si], pbatch['im_file'], validator.save_dir)
    if no_pred: continue

    # Save
    if validator.args.save_json or validator.args.save_txt:
        predn_scaled=validator.scale_preds(predn, pbatch)
    if validator.args.save_json:
        validator.pred_to_json(predn_scaled, pbatch)
    if validator.args.save_txt:
        validator.save_one_txt(predn_scaled, validator.args.save_conf, pbatch['ori_shape'],
                              validator.save_dir/'labels'/f"{Path(pbatch['im_file']).stem}.txt")
    
    break

AttributeError: 'Results' object has no attribute 'masks'